# Gabarito — Exercícios de Agregações (Parte 3) e SQLite (Parte 5)

Resolução comentada dos 10 exercícios propostos em
[08_exercicios_agregacoes_sqlite.ipynb](08_exercicios_agregacoes_sqlite.ipynb).
Cada exercício é resolvido na ordem (alguns dependem do resultado do anterior,
como avisado no enunciado original) e roda de verdade sobre os dados reais do
projeto — não é pseudocódigo.

Este notebook **escreve** em `data/clima.db` (exercícios 7-9 criam as tabelas
`clima_semanal` e `cidades`), da mesma forma que rodar o notebook de exercícios
resolvido escreveria.

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import Column, MetaData, String, Table, create_engine, func, select
from sqlalchemy.dialects.sqlite import insert as sqlite_upsert

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
DB_PATH = DATA_DIR / "clima.db"

horario = pd.read_csv(PROCESSED_DIR / "clima_tratado.csv", parse_dates=["datetime"])
diario = pd.read_csv(PROCESSED_DIR / "clima_diario.csv", parse_dates=["data"])

horario.shape, diario.shape

((3720, 6), (155, 17))

## Exercício 1 — Agregação semanal

Mesma receita do `groupby` diário da Parte 3, trocando `freq="D"` por
`freq="W"` no `pd.Grouper`.

In [2]:
semanal = (
    horario.groupby(["cidade", pd.Grouper(key="datetime", freq="W")])
    .agg(
        temp_media=("temp_c", "mean"),
        temp_min=("temp_c", "min"),
        temp_max=("temp_c", "max"),
        precipitacao_total=("precipitacao_mm", "sum"),
    )
    .reset_index()
    .rename(columns={"datetime": "semana"})
)
semanal["semana"] = semanal["semana"].dt.date

semanal.head(10)

,cidade,semana,temp_media,temp_min,temp_max,precipitacao_total
0,manaus,2025-01-05,26.583333,24.3,30.2,39.9
1,manaus,2025-01-12,27.174702,24.3,30.7,39.3
2,manaus,2025-01-19,27.629464,25.0,31.7,14.2
3,manaus,2025-01-26,26.645833,23.4,31.5,37.9
4,manaus,2025-02-02,27.225417,24.4,30.8,12.2
5,porto_alegre,2025-01-05,24.430417,18.8,32.4,10.3
6,porto_alegre,2025-01-12,23.335714,18.1,30.9,5.8
7,porto_alegre,2025-01-19,26.519048,18.7,35.2,8.1
8,porto_alegre,2025-01-26,26.578571,21.7,36.3,23.5
9,porto_alegre,2025-02-02,24.685417,20.3,32.1,31.3


Confirmando a dica: cada `semana` é rotulada pelo **último** dia daquele
intervalo de 7 dias (um domingo) — a primeira linha de cada cidade,
`2025-01-05`, já é a segunda quarta-feira do mês contada a partir de
01/01/2025 (uma quarta-feira). E a última semana de cada cidade
(`2025-02-02`) tem só 2 dias de dado (31/01 e 01/02 não existem — só
sobra o dado até 31/01), diferente das demais, que cobrem 7 dias inteiros —
dá pra confirmar contando quantas linhas de `horario` caem em cada semana:

In [3]:
horario.groupby(pd.Grouper(key="datetime", freq="W"))["datetime"].count()

datetime
2025-01-05    600
2025-01-12    840
2025-01-19    840
2025-01-26    840
2025-02-02    600
Freq: W-SUN, Name: datetime, dtype: int64

## Exercício 2 — Classificando o vento: `apply` vs. `np.select`

In [4]:
def classificar_vento(vento_kmh: float) -> str:
    if vento_kmh < 10:
        return "calmo"
    elif vento_kmh <= 25:
        return "moderado"
    else:
        return "forte"


horario["categoria_vento"] = horario["vento_kmh"].apply(classificar_vento)

condicoes = [
    horario["vento_kmh"] < 10,
    horario["vento_kmh"] <= 25,
]
valores = ["calmo", "moderado"]
horario["categoria_vento_np"] = np.select(condicoes, valores, default="forte")

print("as duas colunas batem:", (horario["categoria_vento"] == horario["categoria_vento_np"]).all())
horario["categoria_vento"].value_counts()

as duas colunas batem: True


categoria_vento
calmo       2565
moderado    1155
Name: count, dtype: int64

Nenhuma hora do dataset passou de 25 km/h de vento — não há categoria
`"forte"` nos dados reais (a faixa existe no código, só não foi observada
neste mês/nessas cidades). Depois de conferir que as duas colunas concordam,
descartamos a coluna de comparação (mantemos só `categoria_vento`, calculada
com `apply`, como versão "oficial" — mesmo raciocínio do notebook original
para `categoria_temp`) e medimos o tempo das duas abordagens:

In [5]:
horario = horario.drop(columns=["categoria_vento_np"])

In [6]:
%timeit horario["vento_kmh"].apply(classificar_vento)
%timeit np.select(condicoes, valores, default="forte")

372 μs ± 24.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


58.3 μs ± 198 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Exercício 3 — Amplitude térmica e média móvel

In [7]:
diario = diario.sort_values(["cidade", "data"]).reset_index(drop=True)

diario["amplitude_termica"] = diario["temp_max"] - diario["temp_min"]
diario["amplitude_movel_5d"] = diario.groupby("cidade")["amplitude_termica"].transform(
    lambda s: s.rolling(window=5, min_periods=1).mean()
)

diario[["cidade", "data", "temp_min", "temp_max", "amplitude_termica", "amplitude_movel_5d"]].head(7)

,cidade,data,temp_min,temp_max,amplitude_termica,amplitude_movel_5d
0,manaus,2025-01-01,25.5,29.8,4.3,4.300000
1,manaus,2025-01-02,24.7,27.8,3.1,3.700000
2,manaus,2025-01-03,24.9,30.2,5.3,4.233333
3,manaus,2025-01-04,24.5,28.9,4.4,4.275000
4,manaus,2025-01-05,24.3,28.0,3.7,4.160000
5,manaus,2025-01-06,24.3,29.3,5.0,4.300000
6,manaus,2025-01-07,25.1,28.3,3.2,4.320000


Repare a ordenação por `["cidade", "data"]` **antes** do `rolling` — sem ela,
como o CSV já vinha ordenado por cidade e data (Parte 3 salvou assim), o
resultado até coincidiria por acaso aqui, mas o passo continua sendo
obrigatório em geral (não dá pra confiar na ordem de um arquivo lido do
disco).

## Exercício 4 — A cidade mais instável do dia

In [8]:
mais_instavel_do_dia = (
    diario.loc[diario.groupby("data")["amplitude_termica"].idxmax()]
    [["data", "cidade", "amplitude_termica"]]
    .rename(columns={"cidade": "cidade_mais_instavel"})
    .reset_index(drop=True)
)

mais_instavel_do_dia.head(10)

,data,cidade_mais_instavel,amplitude_termica
0,2025-01-01,porto_alegre,10.9
1,2025-01-02,porto_alegre,10.0
2,2025-01-03,rio_de_janeiro,10.8
3,2025-01-04,rio_de_janeiro,9.9
4,2025-01-05,sao_paulo,12.3
5,2025-01-06,sao_paulo,12.1
6,2025-01-07,sao_paulo,10.2
7,2025-01-08,sao_paulo,9.0
8,2025-01-09,porto_alegre,9.5
9,2025-01-10,porto_alegre,11.4


In [9]:
mais_instavel_do_dia["cidade_mais_instavel"].value_counts()

cidade_mais_instavel
porto_alegre      14
sao_paulo         10
rio_de_janeiro     7
Name: count, dtype: int64

São Paulo e Porto Alegre concentram a maioria dos dias de maior amplitude
térmica do mês — faz sentido: são as duas cidades mais distantes do Equador
entre as cinco, com maior variação entre mínima e máxima diárias; Manaus e
Recife, mais próximas da linha do Equador, tendem a ter dias mais
"estáveis" termicamente.

## Exercício 5 — Pivô de chuva e dias de chuva simultânea

In [10]:
pivot_chuva = pd.pivot_table(diario, index="data", columns="cidade", values="precipitacao_total")
pivot_chuva.head()

cidade,manaus,porto_alegre,recife,rio_de_janeiro,sao_paulo
data,,,,,
2025-01-01,11.0,9.1,3.5,3.1,5.1
2025-01-02,7.3,0.8,2.3,0.0,1.4
2025-01-03,1.0,0.0,1.3,1.8,4.3
2025-01-04,11.8,0.0,3.6,5.3,14.4
2025-01-05,8.8,0.4,3.5,0.8,3.6


In [11]:
dias_chuva_simultanea = (pivot_chuva > 0).all(axis=1).sum()
print(f"Dias com chuva nas 5 cidades ao mesmo tempo: {dias_chuva_simultanea} de {len(pivot_chuva)}")

Dias com chuva nas 5 cidades ao mesmo tempo: 17 de 31


## Exercício 6 — `merge` com população e chuva per capita

In [12]:
populacao_meta = pd.DataFrame([
    {"cidade": "sao_paulo", "populacao_mil": 12300},
    {"cidade": "rio_de_janeiro", "populacao_mil": 6700},
    {"cidade": "manaus", "populacao_mil": 2200},
    {"cidade": "porto_alegre", "populacao_mil": 1300},
    {"cidade": "recife", "populacao_mil": 1650},
])

diario = diario.merge(populacao_meta, on="cidade", how="left")
assert diario["populacao_mil"].isna().sum() == 0, "alguma cidade não bateu no merge"

diario["chuva_per_capita"] = diario["precipitacao_total"] / diario["populacao_mil"]

diario.groupby("cidade")["chuva_per_capita"].mean().sort_values(ascending=False)

cidade
recife            0.002141
manaus            0.002104
porto_alegre      0.001960
sao_paulo         0.000779
rio_de_janeiro    0.000307
Name: chuva_per_capita, dtype: float64

Recife e Manaus lideram: choveu proporcionalmente mais para o tamanho da
população dessas cidades do que em São Paulo ou Rio — mesmo Manaus tendo o
maior volume total de chuva (ver `precipitacao_total` na Parte 3), o Rio de
Janeiro tem quase 5x mais população, o que derruba sua chuva *per capita*
para a última posição mesmo com dias de chuva forte pontuais (lembrar do
outlier de `temp_c` do Rio na Parte 2 — cidade com dados mais "extremos" em
geral).

## Exercício 7 — Persistindo a visão semanal no SQLite

*(usa a variável `semanal` do Exercício 1)*

In [13]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS clima_semanal (
    cidade TEXT NOT NULL,
    semana TEXT NOT NULL,
    temp_media REAL,
    temp_min REAL,
    temp_max REAL,
    precipitacao_total REAL,
    PRIMARY KEY (cidade, semana)
)
''')
conn.commit()

In [14]:
def upsert_clima_semanal(df: pd.DataFrame, conn: sqlite3.Connection) -> None:
    linhas = list(
        df[["cidade", "semana", "temp_media", "temp_min", "temp_max", "precipitacao_total"]]
        .assign(semana=lambda d: d["semana"].astype(str))
        .itertuples(index=False, name=None)
    )

    conn.executemany(
        '''
        INSERT INTO clima_semanal (cidade, semana, temp_media, temp_min, temp_max, precipitacao_total)
        VALUES (?, ?, ?, ?, ?, ?)
        ON CONFLICT (cidade, semana) DO UPDATE SET
            temp_media = excluded.temp_media,
            temp_min = excluded.temp_min,
            temp_max = excluded.temp_max,
            precipitacao_total = excluded.precipitacao_total
        ''',
        linhas,
    )
    conn.commit()


upsert_clima_semanal(semanal, conn)
antes = pd.read_sql("SELECT COUNT(*) AS total FROM clima_semanal", conn).iloc[0, 0]

# Rodando de novo sobre o mesmo dado: não deve duplicar nem quebrar por PK
upsert_clima_semanal(semanal, conn)
depois = pd.read_sql("SELECT COUNT(*) AS total FROM clima_semanal", conn).iloc[0, 0]

print(f"linhas antes da 2ª chamada: {antes} | depois: {depois} | idempotente: {antes == depois}")

linhas antes da 2ª chamada: 25 | depois: 25 | idempotente: True


## Exercício 8 — A tabela `cidades` que falta no banco

Uma nuance em relação à dica do notebook de exercícios: `to_sql(...,
if_exists="replace")` sozinho recria a tabela **sem** a `PRIMARY KEY` (o
`to_sql` do pandas não sabe nada sobre constraints, só sobre tipos de coluna)
— e o enunciado pede explicitamente `cidade` como chave primária. Para ter as
duas coisas (tabela sempre recriada do zero + chave primária de verdade),
criamos a tabela manualmente com `CREATE TABLE` e inserimos os dados com
`to_sql(..., if_exists="append")` em vez de `"replace"`.

In [15]:
cidades_meta = pd.DataFrame([
    {"cidade": "sao_paulo", "nome_exibicao": "São Paulo", "uf": "SP", "regiao": "Sudeste"},
    {"cidade": "rio_de_janeiro", "nome_exibicao": "Rio de Janeiro", "uf": "RJ", "regiao": "Sudeste"},
    {"cidade": "manaus", "nome_exibicao": "Manaus", "uf": "AM", "regiao": "Norte"},
    {"cidade": "porto_alegre", "nome_exibicao": "Porto Alegre", "uf": "RS", "regiao": "Sul"},
    {"cidade": "recife", "nome_exibicao": "Recife", "uf": "PE", "regiao": "Nordeste"},
])

cursor.execute("DROP TABLE IF EXISTS cidades")
cursor.execute('''
CREATE TABLE cidades (
    cidade TEXT PRIMARY KEY,
    nome_exibicao TEXT,
    uf TEXT,
    regiao TEXT
)
''')
conn.commit()

cidades_meta.to_sql("cidades", conn, if_exists="append", index=False)

5

In [16]:
dias_chuvosos_por_regiao = pd.read_sql(
    '''
    SELECT c.regiao, COUNT(*) AS dias_chuvosos
    FROM clima_diario AS d
    JOIN cidades AS c ON d.cidade = c.cidade
    WHERE d.categoria_chuva = 'chuvoso'
    GROUP BY c.regiao
    ORDER BY dias_chuvosos DESC
    ''',
    conn,
)
dias_chuvosos_por_regiao

,regiao,dias_chuvosos
0,Sudeste,39
1,Norte,24
2,Nordeste,23
3,Sul,15


In [17]:
conn.close()

## Exercício 9 — A mesma tabela `cidades`, agora com SQLAlchemy Core

Em vez de redeclarar as colunas de `clima_diario` na mão (o exercício só
precisa de `cidade`/`categoria_chuva`, mas a tabela real tem muito mais
colunas), usamos `autoload_with=engine` para que o SQLAlchemy **reflita** o
schema já existente direto do banco — evita duplicar a definição da tabela e
o risco de ela ficar dessincronizada do que existe de verdade no banco.

In [18]:
engine = create_engine(f"sqlite:///{DB_PATH}")
metadata = MetaData()

cidades_sa = Table(
    "cidades",
    metadata,
    Column("cidade", String, primary_key=True),
    Column("nome_exibicao", String),
    Column("uf", String),
    Column("regiao", String),
)

metadata.create_all(engine, checkfirst=True)  # não recria — a tabela já existe (Exercício 8)

clima_diario_sa = Table("clima_diario", metadata, autoload_with=engine)
[c.name for c in clima_diario_sa.columns]

['cidade',
 'data',
 'temp_media',
 'temp_min',
 'temp_max',
 'umidade_media',
 'precipitacao_total',
 'vento_medio',
 'categoria_temp',
 'categoria_chuva',
 'media_movel_3d',
 'media_movel_7d',
 'ranking_temp_dia',
 'indice_conforto_c']

In [19]:
registros = cidades_meta.to_dict(orient="records")

stmt = sqlite_upsert(cidades_sa).values(registros)
stmt = stmt.on_conflict_do_update(
    index_elements=["cidade"],
    set_={
        "nome_exibicao": stmt.excluded.nome_exibicao,
        "uf": stmt.excluded.uf,
        "regiao": stmt.excluded.regiao,
    },
)

with engine.begin() as conn_sa:
    conn_sa.execute(stmt)

In [20]:
query = (
    select(cidades_sa.c.regiao, func.count().label("dias_chuvosos"))
    .select_from(
        clima_diario_sa.join(cidades_sa, clima_diario_sa.c.cidade == cidades_sa.c.cidade)
    )
    .where(clima_diario_sa.c.categoria_chuva == "chuvoso")
    .group_by(cidades_sa.c.regiao)
    .order_by(func.count().desc())
)

pd.read_sql(query, engine)

,regiao,dias_chuvosos
0,Sudeste,39
1,Norte,24
2,Nordeste,23
3,Sul,15


Mesmo resultado do Exercício 8, como esperado — é a mesma pergunta, duas
formas de escrever a query.

## Exercício 10 — Reprocessar e conferir contra o banco

In [21]:
raw_do_banco = pd.read_sql("SELECT * FROM clima_raw", engine, parse_dates=["datetime"])

diario_recalculado = (
    raw_do_banco.groupby(["cidade", pd.Grouper(key="datetime", freq="D")])
    .agg(
        temp_media=("temp_c", "mean"),
        temp_min=("temp_c", "min"),
        temp_max=("temp_c", "max"),
        umidade_media=("umidade_pct", "mean"),
        precipitacao_total=("precipitacao_mm", "sum"),
        vento_medio=("vento_kmh", "mean"),
    )
    .reset_index()
    .rename(columns={"datetime": "data"})
)
diario_recalculado["data"] = diario_recalculado["data"].dt.date

diario_recalculado.head(3)

,cidade,data,temp_media,temp_min,temp_max,umidade_media,precipitacao_total,vento_medio
0,manaus,2025-01-01,27.391667,25.5,29.8,84.125000,11.0,5.654167
1,manaus,2025-01-02,25.908333,24.7,27.8,86.125000,7.3,6.083333
2,manaus,2025-01-03,27.587500,24.9,30.2,80.916667,1.0,4.937500


In [22]:
colunas_para_comparar = [
    "temp_media", "temp_min", "temp_max", "umidade_media", "precipitacao_total", "vento_medio",
]

diario_persistido = pd.read_sql("SELECT * FROM clima_diario", engine)
diario_persistido["data"] = pd.to_datetime(diario_persistido["data"]).dt.date

comparacao = diario_recalculado.merge(
    diario_persistido[["cidade", "data"] + colunas_para_comparar],
    on=["cidade", "data"],
    suffixes=("_recalc", "_banco"),
)

print(f"{len(comparacao)} linhas comparadas (esperado: {len(diario_persistido)})")
for col in colunas_para_comparar:
    bate = np.isclose(comparacao[f"{col}_recalc"], comparacao[f"{col}_banco"]).all()
    print(f"{col}: bate = {bate}")

155 linhas comparadas (esperado: 155)
temp_media: bate = True
temp_min: bate = True
temp_max: bate = True
umidade_media: bate = True
precipitacao_total: bate = True
vento_medio: bate = True


In [23]:
engine.dispose()

Todas as colunas batem: reprocessar `clima_raw` direto do banco, do zero,
com a mesma receita da Parte 3, reproduz exatamente `clima_diario` — o que é
uma boa confirmação de que a persistência da Parte 5 não perdeu nem alterou
informação no caminho entre o CSV tratado e as tabelas do SQLite. Esse tipo
de checagem (reprocessar e comparar) é uma forma simples de teste de
regressão para um pipeline de dados, sem precisar de uma suíte de testes
formal.